# Tilt fix from the matches in the photo view (iteration 44)

**Idea (user, 2026-11-18).** In the photo view the matcher gives about 80 matches spread over the whole photo. A person looking at the overlay would see that the photo does not fit the map perfectly because the camera's tilt was a little different from the metadata, and would correct it. Do the same: every inlier match says which ground point (in map metres) a photo pixel sees; fit the camera pose (position x, y, pitch, roll, yaw, height) so that all photo pixels land on their ground points, with the metadata's noise as a prior (pitch and roll 1 degree, yaw 2 degrees, height 2 %: the noise that reproduced the real error in the synthetic audit of iteration 40).

**Why it may not work (written first).** A narrow image cannot tell a small tilt from a small shift: tilt by d degrees at height h moves the ground point under the image centre by h * d, the same as a position change. Only the perspective part, the scale changing across the picture, separates them. This photo is wide (focal length 202 px on 320 px: 76 degrees), so the effect is about 2.7 % edge to edge for 1 degree, and 80 matches may be enough to see it. The earlier pose-from-matches test (iteration 41) used the few matches of the 224 px canvas and recovered 4 to 11 % of the error on clean synthetic photos.

**Measured here.** For every working photo from 1000 m: XoFTR matches in the photo view at a candidate place 30 to 60 m from the truth (random direction, fixed seed, as a coarse peak would be), 3 heading queries, the query with most inliers (at least 20) is used. Truth (the photo's GPS) is known, so the position error of each estimator is measured directly: V0 median of the match votes (what the pipeline does now), V1 pose fit with the nominal priors, V2 the same with priors 3 times wider, V3 fit of position and height only (no tilt). Reported: median error in sigma, share within 1 sigma, mean points per photo (10/6/3/0 by sigma).

**Kill rule (written before the run).** The tilt fit is useful only if V1 raises the mean points per photo by at least 0.3 over V0 on the photos, with no priors tuned on these photos (the priors are the audit's assumed noise, not fitted), and the gain holds in both halves (odd/even photos).

Env: `EF_HEADLESS, EF_DATASET, EF_STORE, EF_OUT, EF_TAG, EF_LIMIT, EF_MIN_HEIGHT`.

In [ ]:
import os, sys, math
import numpy as np, pandas as pd, torch, cv2
HEADLESS = os.environ.get("EF_HEADLESS", "../../../../Main_Pipeline_Headless"); sys.path.insert(0, os.path.abspath(HEADLESS))
import settings
settings.LEVELS = [3, 4, 5, 6, 7, 8, 9, 10, 11]
from geoloc import photo, queries, scoring, fine_matching, photo_view, native_fine, xoftr_matcher
from geoloc.cli import read_rows
from geoloc.map_store import MapStore
from geoloc.levels import level_gsd
from scipy.optimize import least_squares
DATASET = os.environ["EF_DATASET"]; STORE = os.environ["EF_STORE"]; OUT = os.environ.get("EF_OUT", "results/tilt_fit"); os.makedirs(OUT, exist_ok=True)
TAG = os.environ.get("EF_TAG", ""); MIN_HEIGHT = float(os.environ.get("EF_MIN_HEIGHT", 1000)); LIMIT = int(os.environ.get("EF_LIMIT", 0))
store = MapStore(STORE); rows = read_rows(DATASET)
xoftr = xoftr_matcher.load_xoftr(os.path.join(os.path.abspath(HEADLESS), "models", settings.XOFTR_FILE))
query_list = [q for q in queries.query_list() if abs(q["scale_factor"] - 1.0) < 1e-6]
todo = [i for i, r in enumerate(rows) if float(r["alt"]) >= MIN_HEIGHT]
if LIMIT: todo = todo[:LIMIT]
print(len(todo), "photos", settings.GEOMETRY)

In [ ]:
# phase 1: matches in the photo view at a candidate place near the truth (needs the GPU); saved so the fits can be redone without it
rng = np.random.default_rng(11); saved = []
for index in todo:
    meta = rows[index]; camera = photo.read_camera(meta); pose = photo.read_pose(meta); path = os.path.join(DATASET, "images", meta["filename"])
    truth = np.array(store.gps_to_metres(float(meta["lat_gtruth"]), float(meta["lon_gtruth"]))); angle = rng.uniform(0, 2 * np.pi); distance = rng.uniform(30, 60)
    cand = truth + distance * np.array([np.cos(angle), np.sin(angle)])
    view = photo_view.PhotoView(path, pose, camera, store.levels)
    best = None
    for q in query_list:
        grid = view.grid(q); drawn, share = photo_view.draw_map(store.levels[view.level], view.level, grid, cand[0], cand[1])
        if drawn is None or share < 0.5: continue
        tile = native_fine.clean_tile(drawn, view.tile_sigma)
        pa, pb, conf = fine_matching.loftr_matches(xoftr, view.picture, tile)
        matrix, picked = fine_matching.fit_similarity(pa, pb)
        if matrix is None or picked.sum() < 20: continue
        if best is None or picked.sum() > best["n"]:
            best = dict(n=int(picked.sum()), heading=q["heading"], pa=pa[picked], pb=pb[picked], grid=grid)
    if best is None: continue
    g = best["grid"]; pa, pb = best["pa"], best["pb"]
    ea, na = photo_view.sample(g[0], pa), photo_view.sample(g[1], pa); eb, nb = photo_view.sample(g[0], pb), photo_view.sample(g[1], pb)
    saved.append(dict(photo=index, pose=pose, camera=camera, truth=truth, cand=cand, heading=best["heading"], n=best["n"], pa=pa, pb=pb,
                      votes_e=eb - ea, votes_n=nb - na, gsd=pose["height"] * settings.GEOMETRY["scale"] / camera["fx"]))
    print(index, best["n"], flush=True)
import pickle; pickle.dump(saved, open(os.path.join(OUT, "tilt_matches%s.pkl" % TAG), "wb")); print(len(saved), "photos with >= 20 inliers")

In [ ]:
# phase 2: the estimators. Map metres: x east, y south. A photo pixel sees ground truth + G(p); the drawn map shows cand + G(q) at pixel q.
def ground_at(points, camera, pose, heading, dp, dr, dyaw, dlogh):
    g = settings.GEOMETRY
    intrinsic = np.array([[camera["fx"], 0, camera["cx"]], [0, camera["fy"], camera["cy"]], [0, 0, 1]], float)
    dist = np.array([camera["k1"], camera["k2"], camera["p1"], camera["p2"], camera["k3"]])
    und = cv2.undistortPointsIter(points.reshape(-1, 1, 2).astype(np.float64), intrinsic, dist, None, None, (cv2.TERM_CRITERIA_COUNT + cv2.TERM_CRITERIA_EPS, 50, 1e-10)).reshape(-1, 2)
    pitch = roll = 0.0
    if g["attitude"]: pitch = g["pitch_sign"] * pose["pitch"] + dp; roll = g["roll_sign"] * pose["roll"] + dr
    ax, ay, az = photo.camera_axes(pose["yaw"] + g["yaw_offset"] + heading + dyaw, pitch, roll)
    height = (pose["height"] + g["height_offset"]) * g["scale"] * math.exp(dlogh)
    ray = und[:, :1] * ax + und[:, 1:] * ay + az; d = -height / ray[:, 2]
    return ray[:, 0] * d, ray[:, 1] * d
def estimate(item, variant):
    pose, camera, heading, gsd = item["pose"], item["camera"], item["heading"], item["gsd"]
    pa = item["pa"]
    # absolute ground point of every photo pixel from the map: cand + G(pb) (G(pb) = G(pa) + vote)
    e0, n0 = ground_at(pa, camera, pose, heading, 0, 0, 0, 0)
    mx = item["cand"][0] + e0 + item["votes_e"]; my = item["cand"][1] - (n0 + item["votes_n"])
    start = np.array([np.median(mx - e0), np.median(my + n0)])
    if variant == "V0": return start
    tilt = {"V1": 1.0, "V2": 3.0, "V3": 0.0}[variant]
    prior = np.array([1.0, 1.0, 2.0, 0.02]) * (tilt if tilt > 0 else 1.0)
    def resid(p):
        x, y, dp, dr, dyaw, dlogh = p
        if variant == "V3": dp = dr = dyaw = 0.0
        e, n = ground_at(pa, camera, pose, heading, dp, dr, dyaw, dlogh)
        s = 1.5 * gsd
        r = np.r_[(x + e - mx) / s, (y - n - my) / s]
        pr = np.array([dp / prior[0], dr / prior[1], dyaw / prior[2], dlogh / prior[3]])
        if variant == "V3": pr = np.array([0, 0, 0, dlogh / prior[3]])
        return np.r_[r, pr]
    fit = least_squares(resid, np.r_[start, 0, 0, 0, 0], loss="soft_l1", f_scale=2.0)
    return fit.x[:2]
def points_for(error, height, fx):
    sigma = scoring.sigma_m(height, fx); r = error / sigma
    return 10 if r <= 1 else 6 if r <= 2 else 3 if r <= 4 else 0 if r <= 8 else -20
records = []
for item in saved:
    row = dict(photo=item["photo"], height=item["pose"]["height"], n=item["n"], sigma=scoring.sigma_m(item["pose"]["height"], item["camera"]["fx"]))
    for v in ("V0", "V1", "V2", "V3"):
        xy = estimate(item, v); err = float(np.hypot(*(xy - item["truth"]))); row[v + "_err"] = err; row[v + "_pts"] = points_for(err, item["pose"]["height"], item["camera"]["fx"])
    records.append(row)
tab = pd.DataFrame(records); tab.to_csv(os.path.join(OUT, "tilt_fit%s.csv" % TAG), index=False)
print(len(tab), "photos")
summary = []
for v in ("V0", "V1", "V2", "V3"):
    ratio = tab[v + "_err"] / tab.sigma
    summary.append(dict(variant=v, median_err_sigma=round(ratio.median(), 2), within_1sigma=round((ratio <= 1).mean(), 3), within_2sigma=round((ratio <= 2).mean(), 3), mean_points=round(tab[v + "_pts"].mean(), 3),
                        odd_points=round(tab[tab.photo % 2 == 1][v + "_pts"].mean(), 3), even_points=round(tab[tab.photo % 2 == 0][v + "_pts"].mean(), 3)))
print(pd.DataFrame(summary).to_string(index=False))
for v in ("V1", "V2", "V3"):
    gain = tab[v + "_pts"].mean() - tab["V0_pts"].mean(); halves = [tab[tab.photo % 2 == h][v + "_pts"].mean() - tab[tab.photo % 2 == h]["V0_pts"].mean() for h in (0, 1)]
    print("%s gain over V0: %.3f points per photo (halves %.3f / %.3f)  -> %s" % (v, gain, halves[0], halves[1], "USEFUL" if (gain >= 0.3 and min(halves) > 0) else "killed"))